In [ ]:
import pandas as pd
import numpy as np
# spam=1 label
# not spam =0

In [ ]:
df=pd.read_csv('/content/spam_or_not_spam.csv')

In [ ]:
df.head()

In [ ]:
print(df.info())
print(df.shape)

In [ ]:
#check for null values and drop them
df.dropna(inplace=True)
df.isnull().sum()

In [ ]:
#check for duplicates
df.duplicated().sum()

In [ ]:
#remove duplicates
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [ ]:
df.shape

In [ ]:
#EDA
df['label'].value_counts()

In [ ]:
import matplotlib.pyplot  as plt
plt.pie(df['label'].value_counts(),labels=[0,1],autopct='%0.2f')
plt.show()

In [ ]:
import nltk

In [ ]:
from nltk import word_tokenize,sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
def word_token(text):
  tokens=word_tokenize(text)
  return len(tokens)

In [ ]:
df.info()

In [ ]:
word_token('hello everyone hii true')

In [ ]:
df['word_num']=df['email'].apply(word_token)

In [ ]:
df.head()

In [ ]:
df['num_characters']=df['email'].apply(len)

In [ ]:
df.head()

In [ ]:
def sentence(text):
  sent_token=sent_tokenize(text)
  return len(sent_token)

In [ ]:
df['num_sentences']=df['email'].apply(lambda x:len(sent_tokenize(x)))

In [ ]:
df.head()

In [ ]:
df[df['label']==0][['word_num','num_characters','num_sentences']].describe()

In [ ]:
df[df['label']==1][['word_num','num_characters','num_sentences']].describe()

In [ ]:
import seaborn as sns

In [ ]:
sns.histplot(df[df['label']==0]['word_num'])
sns.histplot(df[df['label']==1]['word_num'],color='red')

In [ ]:
df[['label','word_num','num_characters','num_sentences']].corr()

In [ ]:
sns.heatmap(df[['label','word_num','num_characters','num_sentences']].corr(),annot=True)

In [ ]:
import re
import string

In [ ]:
df['email'][0]

In [ ]:
stop_word=set(stopwords.words('english'))
stop_word

In [ ]:
porter=PorterStemmer()
porter

In [ ]:
def tranform_text(text):
  text=re.sub(r'[%#@]','',text)
  text = re.sub(r'_+', ' ', text)
  text=text.lower()
  text=word_tokenize(text)
  new_text=[]
  for i in text:
    if i not in stop_word and i not in string.punctuation:
      i=porter.stem(i)
      new_text.append(i)


  return  ' '.join(new_text)

In [ ]:
tranform_text('date wed NUMBER aug NUMBER NUMBER NUMBER NUMBER NUMBER from chris garrigues cwg dated NUMBER NUMBERfaNUMBERd deepeddy com message id NUMBER NUMBER tmda deepeddy vircio com i can t reproduce this error for me it is very repeatable like every time without fail this is the debug log of the pick happening NUMBER NUMBER NUMBER pick_it exec pick inbox list lbrace lbrace subject ftp rbrace rbrace NUMBER NUMBER sequence mercury NUMBER NUMBER NUMBER exec pick inbox list lbrace lbrace subject ftp rbrace rbrace NUMBER NUMBER sequence mercury NUMBER NUMBER NUMBER ftoc_pickmsgs NUMBER hit NUMBER NUMBER NUMBER marking NUMBER hits NUMBER NUMBER NUMBER tkerror syntax error in expression int note if i run the pick command by hand delta pick inbox list lbrace lbrace subject ftp rbrace rbrace NUMBER NUMBER sequence mercury NUMBER hit that s where the NUMBER hit comes from obviously the version of nmh i m using is delta pick version pick nmh NUMBER NUMBER NUMBER compiled on URL at sun mar NUMBER NUMBER N')

In [ ]:
df['tranform_email']=df['email'].apply(tranform_text)

In [ ]:
df.head()

In [ ]:
from wordcloud import WordCloud

In [ ]:
wc=WordCloud(width=1000,height=700,min_font_size=15,background_color='white')

In [ ]:
spam_wc=wc.generate(df[df['label']==1]['tranform_email'].str.cat(sep=' '))

In [ ]:
plt.imshow(spam_wc)

In [ ]:
ham_wc=wc.generate(df[df['label']==0]['tranform_email'].str.cat(sep=' '))

In [ ]:
plt.imshow(ham_wc)

In [ ]:
df[df['label']==1]['email']

In [ ]:
word_corpus=[]
for msg in df[df['label']==1]['tranform_email'].tolist():
  for word in msg.split():
    word_corpus.append(word)


In [ ]:
len(word_corpus)

In [ ]:
from collections import Counter

In [ ]:
sns.barplot(
    data=pd.DataFrame(
        Counter(word_corpus).most_common(30),
        columns=['word', 'count']
    ),
    x='count',
    y='word'
)
plt.show()

## **MODEL BUILDING**


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes   import BernoulliNB,GaussianNB,MultinomialNB

In [ ]:
cv=CountVectorizer()

In [ ]:
x=cv.fit_transform(df['tranform_email']).toarray()

In [ ]:
print(len(cv.vocabulary_))

In [ ]:
y=df['label'].values
x.shape

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
gnb=GaussianNB()
bnb=BernoulliNB()
mnb=MultinomialNB()

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [ ]:
gnb.fit(x_train,y_train)
y_pred=gnb.predict(x_test)
print(f'the accuracy score:{accuracy_score(y_test,y_pred)}')
print(f'the confusion matrix:\n{confusion_matrix(y_test,y_pred)}')
print(f'the precision score:{precision_score(y_test,y_pred)}')

In [ ]:
bnb.fit(x_train,y_train)
y_pred_2=bnb.predict(x_test)
print(f'the accuracy score:{accuracy_score(y_test,y_pred_2)}')
print(f'the confusion matrix:\n{confusion_matrix(y_test,y_pred_2)}')
print(f'the precision score:{precision_score(y_test,y_pred_2)}')

In [ ]:
mnb.fit(x_train,y_train)
y_pred_3=mnb.predict(x_test)
print(f'the accuracy score:{accuracy_score(y_test,y_pred_3)}')
print(f'the confusion matrix:\n{confusion_matrix(y_test,y_pred_3)}')
print(f'the precision score:{precision_score(y_test,y_pred_3)}')

In [ ]:
!pip install optuna

In [ ]:
import optuna

In [ ]:
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate

In [ ]:
def objective(trials):
  model_select=trials.suggest_categorical('models',['BNB','MNB','GNB','SVM','ADA_BOOST','GRADIENT_BOOST','RANDOM_FOREST_CLF'])
  # n_estimator=trials.suggest_int('estimator',1,100)
  # depth=trials.suggest_int('depth',1,20)
  # learning_rat=trials.suggest_float('lr',0.0025,0.5)

  if model_select=='BNB':
    model=BernoulliNB()
  elif model_select=='MNB':
    model=MultinomialNB()
  elif model_select=='GNB':
    model=GaussianNB()
  elif model_select=='SVM':
    kernel=trials.suggest_categorical('kernel',['linear','rbf'])
    model=SVC(kernel=kernel)
  elif model_select=='ADA_BOOST':
     n_estimator=trials.suggest_int('ada_estimator',1,100)
     depth=trials.suggest_int('ada_depth',1,20)
     learning_rat=trials.suggest_float('ada_lr',0.0025,0.5)
     estimat=DecisionTreeClassifier(max_depth=depth,random_state=42)
     model=AdaBoostClassifier(n_estimators=n_estimator,learning_rate=learning_rat,estimator=estimat)
  elif model_select=='GRADIENT_BOOST':
     n_estimator=trials.suggest_int('boost_estimator',1,100)
     depth=trials.suggest_int('boost_depth',1,20)
     learning_rat=trials.suggest_float('boost_lr',0.0025,0.5)
     model=GradientBoostingClassifier(learning_rate=learning_rat,n_estimators=n_estimator,max_depth=depth)
  elif model_select=='RANDOM_FOREST_CLF':
     n_estimator=trials.suggest_int('forest_estimator',1,100)
     depth=trials.suggest_int('forest_depth',1,20)
     model=RandomForestClassifier(max_depth=depth,n_estimators=n_estimator)

  cross_results=cross_validate(model,x_train,y_train,cv=3,scoring=['accuracy','precision'],n_jobs=-1)

  # model.fit(x_train,y_train)
  # y_pred=model.predict(x_test)
  # matrix=confusion_matrix(y_test,y_pred)
  accuracy = cross_results['test_accuracy'].mean()
  precision = cross_results['test_precision'].mean()


  # score=(
  #     0.20*accuracy+
  #     0.80*precision_scor

  # )
  return accuracy,precision

In [ ]:
study=optuna.create_study(directions=['maximize','maximize'])
study.optimize(objective,n_trials=25)

In [ ]:
results=study.trials_dataframe()
results = results[
    [
        'number',
        'values_0',
        'values_1',
        'params_models'
    ]
]

results = results.rename(
    columns={
        'number': 'Trial',
        'values_0': 'Accuracy',
        'values_1': 'Precision',
        'params_models': 'Model'
    }
)

results = results.sort_values(
    by='Precision',
    ascending=False)
results

In [ ]:
results[(results['Accuracy']>0.97) & (results['Precision']>=0.97)].iloc[0]

In [ ]:
mnb_result=results[results['Model']=='MNB'].sort_values('Precision',ascending=False).iloc[0]

In [ ]:
mnb_trail=[trail for trail in study.trials if trail.params['models']=='MNB']

In [ ]:
for trial in mnb_trail:
  print(trial.number,
        trial.values,
        trial.params
    )

In [ ]:
plt.figure(figsize=(12,5))
sns.barplot(data=results,x=results['Model'],y=results['Precision'])

In [ ]:
plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=results,
    x='Accuracy',
    y='Precision',
    hue='Model',
    s=150
)

plt.title('Accuracy vs Precision')
plt.tight_layout()
plt.show()

In [ ]:
for trail in study.best_trials:
  print('trail',trail.number)
  print('precision values',trail.values[1])
  print('Accuracy values',trail.values[0])
  print('parameter',trail.params)
  print('trail_number',trail.number)
  print('-'*50)

In [ ]:
import optuna.visualization

In [ ]:
from optuna.visualization import _timeline,_optimization_history,plot_param_importances,plot_rank,plot_optimization_history,plot_intermediate_values,plot_param_importances,plot_timeline

In [ ]:
plot_optimization_history(
    study,
    target=lambda t:t.values[0],
    target_name='Accuracy score'
).show()

In [ ]:
plot_optimization_history(
    study,
    target=lambda t:t.values[1],
    target_name='precision_score'
).show()

In [ ]:
plot_timeline(study).show()

In [ ]:
best_trail=max(study.best_trials,key=lambda t:t.values[1])
best_trail

In [ ]:
def create_final_model(params):
  model_select=params['models']
  if model_select=='BNB':
    models=BernoulliNB()

  elif model_select == 'MNB':

        models = MultinomialNB()

  elif model_select == 'GNB':

        models = GaussianNB()

  elif model_select == 'SVM':

        models = SVC(
            kernel=params['kernel']
        )
  elif model_select=='ADA_BOOST':
     estimat=DecisionTreeClassifier(max_depth=params['ada_depth'],random_state=42)
     models=AdaBoostClassifier(n_estimators=params['ada_estimator'],learning_rate=params['ada_lr'],estimator=estimat)


  elif model_select=='RANDOM_FOREST_CLF':
     models=RandomForestClassifier(max_depth=params['forest_depth'],n_estimators=params['forest_estimator'])

  elif model_select == 'GRADIENT_BOOST':

        models = GradientBoostingClassifier(
            learning_rate=params['boost_lr'],
            n_estimators=params['boost_estimator'],
            max_depth=params['boost_depth']
        )

  return models

In [ ]:
final_model=create_final_model(
    best_trail.params
)

In [ ]:
final_model

In [ ]:
final_model.fit(x_train,y_train)
y_pred_final=final_model.predict(x_test)

In [ ]:
print('accuracy_score:',accuracy_score(y_test,y_pred_final))
print('precision_score:',precision_score(y_test,y_pred_final))
print('confusion_matrix:\n',confusion_matrix(y_test,y_pred_final))

In [ ]:
def predict(input):
  tranform_input=tranform_text(input)
  a=cv.transform([tranform_input]).toarray()
  prediction=final_model.predict(a)
  if prediction[0]==1:
    print('its a spam')
  else:
    print(' NOT A SPAM')


In [ ]:
predict("Congratulations! You have won $1000. Claim your prize immediately!")

In [ ]:
final_mnb=MultinomialNB()
final_mnb.fit(x_train,y_train)
y_pred_mnb=final_mnb.predict(x_test)
print('accuracy_score:',accuracy_score(y_test,y_pred_mnb))
print('precision_score:',precision_score(y_test,y_pred_mnb))
print('confusion_matrix:\n',confusion_matrix(y_test,y_pred_mnb))

In [ ]:
def predict_mnb(input):

    transform_input = tranform_text(input)

    a = cv.transform([transform_input]).toarray()

    prediction = final_mnb.predict(a)

    print("Prediction:", prediction)

    if prediction[0] == 1:
        print("SPAM")
    else:
        print("NOT SPAM")

In [ ]:
predict_mnb('Hello, the assignment submission deadline has been extended to Friday. Please submit your completed work before 5 PM.')

In [ ]:
import joblib

In [ ]:
joblib.dump(final_mnb, "spam_mnb_model.pkl")

In [ ]:
joblib.dump(final_model,'random_fores_spam_clf.pkl')

In [ ]:
joblib.dump(cv, "count_vectorizer.pkl")